# Artefato 2: Modelo Clássico Baseline (Machine Learning Tradicional)


## 📋 Objetivo

Este notebook implementa um **modelo baseline** utilizando técnicas de **Machine Learning clássico** para classificação de imagens de satélite. O objetivo é estabelecer uma linha de base de desempenho que servirá como referência para comparação com modelos mais avançados (como redes neurais profundas).

### Características do Modelo Baseline:
- **Tipo**: Classificação binária (positivo/negativo)
- **Algoritmo**: Random Forest Classifier
- **Features**: Valores de pixels das bandas espectrais
- **Pré-processamento**: Normalização com StandardScaler

### Por que um Baseline Clássico?
1. **Simplicidade**: Modelos clássicos são mais fáceis de interpretar e debugar
2. **Rapidez**: Treinamento muito mais rápido que deep learning
3. **Referência**: Estabelece um patamar mínimo de desempenho
4. **Validação**: Confirma que os dados estão bem preparados antes de investir em modelos complexos


## 📦 Importações

Bibliotecas necessárias para manipulação de dados e processamento:


In [3]:
import pandas as pd
import numpy as np
import json
import re

## 📊 Funções de Métricas

Implementação de funções para avaliar o desempenho dos modelos:

### Métricas de Classificação:
- **Accuracy**: Proporção de predições corretas
- **Precision**: Proporção de verdadeiros positivos entre as predições positivas
- **Recall**: Proporção de verdadeiros positivos identificados corretamente
- **F1-Score**: Média harmônica entre precision e recall
- **ROC-AUC**: Área sob a curva ROC (quando probabilidades estão disponíveis)

### Métricas de Regressão:
- **MAE**: Mean Absolute Error
- **RMSE**: Root Mean Squared Error
- **R²**: Coeficiente de determinação


In [4]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    roc_auc_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np


def classification_metrics(y_true, y_pred, y_prob=None):
    results = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

    if y_prob is not None:
        try:
            results["roc_auc"] = roc_auc_score(y_true, y_prob)
        except:
            results["roc_auc"] = None

    return results


def regression_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": rmse,
        "r2": r2_score(y_true, y_pred)
    }

## 🔧 Pipelines de Treinamento

Implementação de pipelines modulares para diferentes tipos de modelos:

### Pipeline de Classificação:
Suporta três algoritmos:
1. **SVM (Support Vector Machine)**: Eficaz para dados de alta dimensionalidade
2. **Random Forest**: Ensemble de árvores de decisão, robusto e interpretável
3. **Logistic Regression**: Modelo linear simples e rápido

### Pipeline de Regressão:
Incluído para uso futuro, caso seja necessário prever valores contínuos.

### Estrutura do Pipeline:
1. **StandardScaler**: Normaliza as features (média=0, desvio padrão=1)
2. **Modelo**: Algoritmo de ML selecionado

Esta estrutura garante que os dados sejam sempre pré-processados de forma consistente.


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split


def create_classifier_pipeline(model_name: str, **kwargs):
    """
    Pipeline para modelos de classificação

    :param model_name: Selecione o modelo svm, random_forest ou logisticregression
    :type model_name: str
    """
    if model_name == "svm":
        model = SVC(**kwargs)
    elif model_name == "random_forest":
        model = RandomForestClassifier(**kwargs)
    elif model_name == "logisticregression":
        model = LogisticRegression(**kwargs)
    else:
        raise ValueError("Modelo de classificação não suportado")

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    return pipeline


def train_classification(X, y, model_name="random_forest", **model_params):
    """ Use para treinar os modelos de classificação, selecionado previamente"""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    pipeline = create_classifier_pipeline(model_name, **model_params)

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    if hasattr(pipeline["model"], "predict_proba"):
        y_prob = pipeline.predict_proba(X_test)[:, 1]
    else:
        y_prob = None

    metrics = classification_metrics(y_test, y_pred, y_prob)

    return pipeline, metrics



## apenas para o caso de decidirmos usar regressão (no futuro)##


def create_regressor_pipeline(model_name: str):
    if model_name == "svm":
        model = SVR()
    elif model_name == "random_forest":
        model = RandomForestRegressor(n_estimators=200, random_state=42)
    elif model_name == "linear":
        model = LinearRegression()
    else:
        raise ValueError("Modelo de regressão não suportado")

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    return pipeline


def train_regression(X, y, model_name="random_forest"):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    pipeline = create_regressor_pipeline(model_name)
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    metrics = regression_metrics(y_test, y_pred)

    return pipeline, metrics

## 🚀 Treinamento do Modelo

### Configuração do Modelo:
- **Algoritmo**: Random Forest
- **Número de árvores**: 300 (n_estimators)
- **Random state**: 42 (para reprodutibilidade)

### Preparação dos Dados:
A função `prepare_dataset` realiza as seguintes etapas:
1. **Carrega o dataset de pixels**: CSV com valores de pixels de cada banda espectral
2. **Extrai o ID da imagem**: Usando regex para identificar o código da imagem no path
3. **Cria labels**: 
   - `1` para imagens positivas (com a característica de interesse)
   - `0` para imagens negativas
   - `-1` para imagens não classificadas (removidas do treinamento)
4. **Separa features (X) e labels (y)**:
   - Remove colunas não relacionadas às bandas espectrais
   - Mantém apenas os valores de pixels como features

### Divisão dos Dados:
- **80%** para treinamento
- **20%** para teste
- Random state fixo para garantir reprodutibilidade


In [7]:
# Constants for model and data paths
MODEL_NAME = "random_forest"
MODEL_PARAMS = {
    "n_estimators": 300,
    "random_state": 42
}
PIXELS_DATASET_PATH = "../../data/pixels_dataset.csv"
EXTRACTED_CODES_PATH = "../../data/extracted_codes.json"

def prepare_dataset(dataset_path: str, extracted_codes_path: str):
    """
    Prepares the dataset by loading the pixel file, extracting image ID,
    creating the 'label' column, and returning features (X) and labels (y).

    :param dataset_path: Path to the pixel dataset CSV file.
    :param extracted_codes_path: Path to the JSON file containing positive and negative codes.
    :return: X (DataFrame of features), y (Series of labels)
    """
    df = pd.read_csv(dataset_path)

    # Load extracted codes
    with open(extracted_codes_path, 'r') as f:
        extracted_codes = json.load(f)

    # Extract image ID (re-using the logic from previous steps)
    all_ids = extracted_codes['negativos'] + extracted_codes['positivos']
    all_ids_sorted = sorted(all_ids, key=len, reverse=True)
    id_pattern = '|'.join(re.escape(id_val) for id_val in all_ids_sorted)
    regex = rf'({id_pattern})'
    df['image_id'] = df['path'].str.extract(regex)[0]

    # Create label column
    df['label'] = df['image_id'].apply(
        lambda x: 1 if x in extracted_codes['positivos'] else (
            0 if x in extracted_codes['negativos'] else -1
        )
    )

    # Drop non-feature columns and separate X and y
    X = df.drop(
        columns=[
            'path', 'filename', 'count', 'height', 'width', 'dtype',
            'crs', 'transform', 'image_id', 'label'
        ]
    )
    y = df["label"]

    return X, y


if __name__ == "__main__":
    print("Preparing data for training...")
    X, y = prepare_dataset(PIXELS_DATASET_PATH, EXTRACTED_CODES_PATH)

    print("Treinando modelo...")
    pipeline, metrics = train_classification(
        X,
        y,
        model_name=MODEL_NAME,
        **MODEL_PARAMS
    )

    print("\nResultados:")
    for k, v in metrics.items():
        print(f"{k}: {v}")

Preparing data for training...


/var/folders/b5/9vybwgfd3tl59r34lly54fjw0000gn/T/ipykernel_68649/582085288.py:30: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['image_id'] = df['path'].str.extract(regex)[0]
/var/folders/b5/9vybwgfd3tl59r34lly54fjw0000gn/T/ipykernel_68649/582085288.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['label'] = df['image_id'].apply(


Treinando modelo...

Resultados:
accuracy: 0.847457627118644
precision: 0.7727272727272727
recall: 0.8095238095238095
f1: 0.7906976744186046
roc_auc: 0.906641604010025


## 📈 Análise dos Resultados

### Métricas Obtidas:
- **Accuracy**: 84.75% - O modelo acerta cerca de 85% das predições
- **Precision**: 77.27% - Das imagens classificadas como positivas, 77% realmente são positivas
- **Recall**: 80.95% - O modelo identifica corretamente 81% das imagens positivas
- **F1-Score**: 79.07% - Boa média harmônica entre precision e recall
- **ROC-AUC**: 90.66% - Excelente capacidade de discriminação entre classes

### Interpretação:
✅ **Pontos Positivos**:
- ROC-AUC acima de 90% indica que o modelo tem boa capacidade de separar as classes
- Recall alto (81%) significa que o modelo não perde muitos casos positivos
- Resultados balanceados entre precision e recall

⚠️ **Pontos de Atenção**:
- Precision de 77% indica que há alguns falsos positivos (23% das predições positivas estão erradas)
- Há espaço para melhoria, especialmente com modelos mais sofisticados

### Próximos Passos:
1. **Feature Engineering**: Explorar features derivadas (índices espectrais, texturas)
2. **Hyperparameter Tuning**: Otimizar parâmetros do Random Forest
3. **Modelos Avançados**: Testar redes neurais convolucionais (CNNs)
4. **Análise de Erros**: Investigar casos de falsos positivos e falsos negativos


## 🎯 Conclusão

Este modelo baseline estabelece uma **linha de base sólida** para o projeto:

1. **Validação dos Dados**: Os resultados confirmam que os dados estão bem preparados e contêm informação suficiente para classificação

2. **Benchmark**: Com ~85% de accuracy e ~91% de ROC-AUC, temos um patamar claro para superar com modelos mais avançados

3. **Rapidez**: O treinamento é rápido, permitindo iterações ágeis durante o desenvolvimento

4. **Interpretabilidade**: Random Forest permite análise de importância de features, útil para entender quais bandas espectrais são mais relevantes